## Human Posture Detection

[<img src ="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" align="left">](https://colab.research.google.com/github/spmallick/learnopencv/blob/master/Posture-analysis-system-using-MediaPipe-Pose/human_posture_analysis.ipynb)

Poor posture is a modern-day epidemic. Research has shown that children as young as 10 years of age are demonstrating spinal degeneration on x-ray. Certain postures, like forward head posture, have been linked to migraines, high blood pressure, and decreased lung capacity. In this notebook, you will learn to create an application to detect posture using mediapipe. We are going to use side view samples to perform analysis and draw conclusion. Practical application would require an webcam pointed at the side view of the person.

### Workflow
 - Find point of interests (landmarks) to check angle.
 - Perform analysis on standard sample images.
 - Find threshold range for good and bad posture.
 - Apply on video/webcam input.

### Goal
To detect neck inclination and torso inclination as shown below.
<br>
<img src = "https://learnopencv.com/wp-content/uploads/2022/03/mp-pose-03-posture-sitting-scaled.jpg" align='center'>

In [1]:
if 'google.colab' in str(get_ipython()):
    !pip install opencv-python
    !pip install mediapipe
else:
    pass

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 566.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Uninstalling protobuf-5.29.4:
      Successfully uninstalled protobuf-5.29.4
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.6 which is incompatible.


## Import Libraries

In [ ]:
!pip install numpy==2.0.2 --force-reinstall
import os
os.kill(os.getpid(), 9)

  Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 36.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.0.2 which is incompatible.


In [1]:
import cv2
import time
import math as m
import mediapipe as mp

In [2]:
# Initilize medipipe selfie segmentation class.
mp_pose = mp.solutions.pose
mp_holistic = mp.solutions.holistic

## Function to calculate offset distance
The setup requires the person to be in proper side view. **`findDistance`** function helps us determine offset distance between two poinst. It can be the hip points, the eyes or shoulder. As these points are always more or less symmetric about the central axis. With this, we are going to incorporate camera alignment assistance in the script. Distnace is calculated using the distance formula.


\begin{align}
distance =  \sqrt{(x2 - x1)^2+(y2 - y1)^2}
\end{align}

In [3]:
def findDistance(x1, y1, x2, y2):
    dist = m.sqrt((x2-x1)**2+(y2-y1)**2)
    return dist

## Function to calculate angle subtended by the line of interest to y-axis
This is the primary deterministic factor for the posture. Using the angle subtended by the neck line and the torso line to y-axis. The neck line connects the shoulder and the eye, here we take shoulder as the pivotal point. Similarly the torso line connects hip and the shoulder, where hip is considered pivotal point.
<br>

<img src="https://learnopencv.com/wp-content/uploads/2022/03/mp-pose-05-neckline-inclination.jpg" alt="MediaPipe pose neck inclination" align="middle" width="500" height="600">

<br>

Taking neck line as an example, here points are $P_1(x_1, y_1)$(shoulder), $P_2(x_2, y_2)$ (eye) and $P_3(x_3, y_3)$ (any point on vertical axis passing through $P_1$).
<br>
Hence, for $P_3$ x-coordinate is same as to that of $P_1$ and since $y_3$ is valid for all y, let's take $y_3 = 0$ for simplicity. <br>To find the inner angle of three points, we take vector approach. Angle between two vectors $\vec{P_{12}}$ and $\vec{P_{13}}$ is given by,
\begin{align}
\theta = \arccos (\frac{\vec{P_{12}}.\vec{P_{13}}}{|\vec{P_{12}}|.|\vec{P_{13}}|})
\end{align}
Solving for $\theta$ we get,
\begin{align}
\theta = \arccos (\frac{y_1^2 - y_1.y_2}{y_1\sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}})
\end{align}

In [4]:
# Calculate angle.
def findAngle(x1, y1, x2, y2):
    theta = m.acos((y2 -y1)*(-y1) / (m.sqrt((x2 - x1)**2 + (y2 - y1)**2) * y1))
    degree = int(180/m.pi)*theta
    return degree

## Function to send alert
Use this function to send alerts when bad posture is detected. Feel free to get creative and customize as per your convenience. You can use telegram notifiction alert system, [chek out Telegram bot automation here](https://core.telegram.org/bots). Or you can take it up a notch by creating an android app.

In [28]:
def sendWarning(message):
    print(message)

## Constants and Initializations

In [29]:
# Initilize frame counters.
good_frames = 0
bad_frames = 0

# Font type.
font = cv2.FONT_HERSHEY_SIMPLEX

# Colors.
blue = (255, 127, 0)
red = (50, 50, 255)
green = (127, 255, 0)
dark_blue = (127, 20, 0)
light_green = (127, 233, 100)
yellow = (0, 255, 255)
pink = (255, 0, 255)

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
!ls "/content/drive/My Drive/BearHack"


output_bueno.mp4  output_more_features.mp4  output_print.mp4	     video_doble_hombro.mp4
output_mix.mp4	  output.mp4		    video_buena_postura.mp4  video_postura.mp4


In [32]:
# Initialize mediapipe pose class.
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# Initialize video capture object.
# For webcam input replace file name with 0.
file_name = '/content/drive/My Drive/BearHack/video_postura.mp4'
cap = cv2.VideoCapture(file_name)

# Meta.
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Initialize video writer.
video_output = cv2.VideoWriter("/content/drive/My Drive/BearHack/output_print.mp4", fourcc, fps, frame_size)

## Processing

Processing the image using mediapipe is fairly simple, after setting minimum detection confidence and minimum tracking confidence, we need to pass the RGB image to `pose.process()` method. The documentation can be found in this [**link**](https://google.github.io/mediapipe/solutions/pose). With the acquired result, we find the coordinates of specific landmarks. Then we find the angles suntended by the line of interset to y-axis and draw conclusion on the basis of results obtained from analysis script. The code is self explanatory.

In [33]:
# Thresholds
NECK_THRESHOLD = 50
TORSO_THRESHOLD = 10

# Timer
last_warning_time = time.time()

print('Processing..')
while cap.isOpened():
    success, image = cap.read()
    if not success:
        print("Null.Frames")
        break

    fps = cap.get(cv2.CAP_PROP_FPS)
    h, w = image.shape[:2]
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    keypoints = pose.process(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    lm = keypoints.pose_landmarks
    if not lm:
        continue

    lmPose = mp_pose.PoseLandmark

    l_shldr = (int(lm.landmark[lmPose.LEFT_SHOULDER].x * w), int(lm.landmark[lmPose.LEFT_SHOULDER].y * h))
    r_shldr = (int(lm.landmark[lmPose.RIGHT_SHOULDER].x * w), int(lm.landmark[lmPose.RIGHT_SHOULDER].y * h))
    l_hip = (int(lm.landmark[lmPose.LEFT_HIP].x * w), int(lm.landmark[lmPose.LEFT_HIP].y * h))
    l_ear = (int(lm.landmark[lmPose.LEFT_EAR].x * w), int(lm.landmark[lmPose.LEFT_EAR].y * h))
    l_knee = (int(lm.landmark[lmPose.LEFT_KNEE].x * w), int(lm.landmark[lmPose.LEFT_KNEE].y * h))
    l_ankle = (int(lm.landmark[lmPose.LEFT_ANKLE].x * w), int(lm.landmark[lmPose.LEFT_ANKLE].y * h))
    mid_back = ((l_shldr[0] + l_hip[0]) // 2, (l_shldr[1] + l_hip[1]) // 2)

    offset = findDistance(l_shldr[0], l_shldr[1], r_shldr[0], r_shldr[1])
    cv2.putText(image,
                str(int(offset)) + (' Aligned' if offset < 100 else ' Not Aligned'),
                (w - 220, 30), font, 0.9, green if offset < 100 else red, 2)

    # Ángulos
    neck_inclination = findAngle(l_shldr[0], l_shldr[1], l_ear[0], l_ear[1])
    torso_inclination = findAngle(l_hip[0], l_hip[1], l_shldr[0], l_shldr[1])
    back_angle = findAngle(l_shldr[0], l_shldr[1], mid_back[0], mid_back[1])
    hip_angle = findAngle(l_hip[0], l_hip[1], mid_back[0], mid_back[1])
    leg_angle = findAngle(l_knee[0], l_knee[1], l_hip[0], l_hip[1])

    # Dibujar puntos
    for pt in [l_shldr, r_shldr, l_ear, l_hip, l_knee, l_ankle, mid_back]:
        cv2.circle(image, pt, 7, yellow, -1)

    # Dibujar líneas
    lines = [(l_shldr, l_ear), (l_shldr, (l_shldr[0], l_shldr[1] - 100)),
             (l_hip, l_shldr), (l_hip, (l_hip[0], l_hip[1] - 100)),
             (mid_back, l_shldr), (mid_back, l_hip),
             (l_hip, l_knee), (l_knee, l_ankle)]

    angle_text_string = f'Neck: {int(neck_inclination)}  Torso: {int(torso_inclination)}  Back: {int(back_angle)}  Hip: {int(hip_angle)}  Knee: {int(leg_angle)}'

    # Evaluación de postura
    if neck_inclination < NECK_THRESHOLD and torso_inclination < TORSO_THRESHOLD:
        bad_frames = 0
        good_frames += 1
        cv2.putText(image, angle_text_string, (10, 30), font, 0.7, light_green, 2)
        for a, b in lines:
            cv2.line(image, a, b, green, 4)
        feedback = "Tu postura es correcta"
    else:
        good_frames = 0
        bad_frames += 1
        cv2.putText(image, angle_text_string, (10, 30), font, 0.7, red, 2)
        for a, b in lines:
            cv2.line(image, a, b, red, 4)

        # Evaluar mensaje
        if neck_inclination >= NECK_THRESHOLD and torso_inclination >= TORSO_THRESHOLD:
            feedback = "Tu postura está dañando tu salud gravemente"
        elif neck_inclination >= NECK_THRESHOLD:
            feedback = "Recoloca el cuello. Cambia tu postura"
        elif torso_inclination >= TORSO_THRESHOLD:
            feedback = "Recoloca la espalda. Cambia tu postura"

    # Mostrar mensaje cada 3 segundos
    current_time = time.time()
    if current_time - last_warning_time >= 3:
        sendWarning(feedback)
        last_warning_time = current_time

    # Mostrar mensaje en pantalla
    cv2.putText(image, feedback, (10, 60), font, 0.8, pink, 2)

    # Mostrar tiempo
    good_time = (1 / fps) * good_frames
    bad_time = (1 / fps) * bad_frames
    if good_time > 0:
        cv2.putText(image, f'Good Posture Time: {round(good_time, 1)}s', (10, h - 20), font, 0.8, green, 2)
    else:
        cv2.putText(image, f'Bad Posture Time: {round(bad_time, 1)}s', (10, h - 20), font, 0.8, red, 2)

    if bad_time > 30:
        sendWarning("Llevas más de 30 segundos con mala postura, corrige inmediatamente")

    video_output.write(image)

print('Finished.')
cap.release()
video_output.release()

Processing..
Tu postura es correcta
Tu postura es correcta
Tu postura es correcta
Tu postura es correcta
Tu postura es correcta
Tu postura es correcta
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Tu postura está dañando tu salud gravemente
Recoloca la espalda. Cambia tu postura
Tu postura está dañando tu salud gravemente
Recoloca la espalda. Cambia tu postura
Recoloca la espalda. Cambia tu postura
Recoloca la espalda. Cambia tu postura
Null.Frames
Finished.
